In [3]:
from nats_bench import create
from nats_bench.api_utils import time_string
import numpy as np

# Create the API for size search space
api = create("/home/ankilab/hw-nats-bench/benchmarks/NATS-tss-v1_0-3ffb9.pickle.pbz2", 'tss', fast_mode=False, verbose=False)

In [4]:
print('{:} There are {:} architectures on the size search space'.format(time_string(), len(api)))

# Obtain the 12-th candidate's configureation on CIFAR-10
config = api.get_net_config(12, 'cifar10')
print(config)

[2026-04-08 12:25:41] There are 15625 architectures on the size search space
{'name': 'infer.tiny', 'C': 16, 'N': 5, 'arch_str': '|none~0|+|none~0|none~1|+|none~0|nor_conv_3x3~1|avg_pool_3x3~2|', 'num_classes': 10}


In [5]:
import xautodl  # import this lib -- "https://github.com/D-X-Y/AutoDL-Projects", you can use pip install xautodl
from xautodl.models import get_cell_based_tiny_net
# create the network, which is the sub-class of torch.nn.Module
network = get_cell_based_tiny_net(config)

In [6]:
print(network)

TinyNetwork(
  TinyNetwork(C=16, N=5, L=17)
  (stem): Sequential(
    (0): Conv2d(3, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
    (1): BatchNorm2d(16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  )
  (cells): ModuleList(
    (0-4): 5 x InferCell(
      info :: nodes=4, inC=16, outC=16, [1<-(I0-L0) | 2<-(I0-L1,I1-L2) | 3<-(I0-L3,I1-L4,I2-L5)], |none~0|+|none~0|none~1|+|none~0|nor_conv_3x3~1|avg_pool_3x3~2|
      (layers): ModuleList(
        (0-3): 4 x Zero(C_in=16, C_out=16, stride=1)
        (4): ReLUConvBN(
          (op): Sequential(
            (0): ReLU()
            (1): Conv2d(16, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
            (2): BatchNorm2d(16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
          )
        )
        (5): POOLING(
          (op): AvgPool2d(kernel_size=3, stride=1, padding=1)
        )
      )
    )
    (5): ResNetBasicblock(
      ResNetBasicblock(inC=16, outC=3

In [7]:
from xautodl.utils import count_parameters_in_MB
print('The model parameters are {:} MB'.format(count_parameters_in_MB(network)))

The model parameters are 0.316346 MB


In [8]:
# --- PyTorch → ONNX → TFLite conversion pipeline ---
# Requires: onnx2tf (converts ONNX to TFLite)
# Install if not present:
# !pip install onnx2tf

import os
import torch
import onnx
import onnx2tf

# ── 1. Export PyTorch model to ONNX ──────────────────────────────────────────
network.eval()
# CIFAR-10 input shape: (batch=1, channels=3, height=32, width=32)
dummy_input = torch.randn(1, 3, 32, 32)

onnx_path = "/tmp/nats_network.onnx"
torch.onnx.export(
    network,
    dummy_input,
    onnx_path,
    opset_version=11,
    input_names=["input"],
    output_names=["output"],
    dynamic_axes={"input": {0: "batch_size"}, "output": {0: "batch_size"}},
)

# Verify the exported ONNX model
onnx_model = onnx.load(onnx_path)
onnx.checker.check_model(onnx_model)
print(f"ONNX model saved and verified: {onnx_path}")

# ── 2. Convert ONNX → TFLite via onnx2tf ─────────────────────────────────────
tflite_out_dir = "/tmp/nats_tflite"
onnx2tf.convert(
    input_onnx_file_path=onnx_path,
    output_folder_path=tflite_out_dir,
    non_verbose=True,
)

# Locate the generated .tflite file
tflite_files = [f for f in os.listdir(tflite_out_dir) if f.endswith(".tflite")]
if tflite_files:
    tflite_path = os.path.join(tflite_out_dir, tflite_files[0])
    size_kb = os.path.getsize(tflite_path) / 1024
    print(f"TFLite model saved: {tflite_path}  ({size_kb:.1f} KB)")
else:
    print("No .tflite file found — check onnx2tf output above.")


E0000 00:00:1775651176.652056 2745172 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1775651177.419550 2745172 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1775651177.557277 2745172 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1775651177.557296 2745172 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1775651177.557298 2745172 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1775651177.557301 2745172 computation_placer.cc:177] computation placer already registered. Please check linka

ModuleNotFoundError: No module named 'onnxscript'

In [1]:
import tensorflow as tf
from tensorflow.keras import layers
from nats_bench import create


# -----------------------------
# Operation definitions
# -----------------------------

def op_none(x, C):
    return x * 0

def op_skip(x, C):
    return x

def op_conv1x1(x, C):
    x = layers.Conv2D(C, 1, padding="same", use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    return layers.ReLU()(x)

def op_conv3x3(x, C):
    x = layers.Conv2D(C, 3, padding="same", use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    return layers.ReLU()(x)

def op_avgpool(x, C):
    return layers.AveragePooling2D(3, strides=1, padding="same")(x)


OPS = {
    "none": op_none,
    "skip_connect": op_skip,
    "nor_conv_1x1": op_conv1x1,
    "nor_conv_3x3": op_conv3x3,
    "avg_pool_3x3": op_avgpool
}


# -----------------------------
# Parse NATS architecture
# -----------------------------

def parse_arch(arch_str):

    nodes = arch_str.split('+')
    arch = []

    for node in nodes:
        node = node.strip('|')
        edges = node.split('|')

        parsed_edges = []

        for e in edges:
            if not e:
                continue

            op, src = e.split('~')
            parsed_edges.append((op, int(src)))

        arch.append(parsed_edges)

    return arch


# -----------------------------
# Build NATS cell
# -----------------------------

def build_cell(x, arch, C):

    states = [x]

    for edges in arch:

        node_inputs = []

        for op_name, src in edges:

            op = OPS[op_name]
            h = op(states[src], C)

            node_inputs.append(h)

        node = layers.Add()(node_inputs)
        states.append(node)

    out = layers.Concatenate()(states[1:])
    out = layers.Conv2D(C, 1, padding="same")(out)

    return out


# -----------------------------
# Build full CIFAR network
# -----------------------------

def build_model(arch_str, C=16, N=5):

    arch = parse_arch(arch_str)

    inputs = tf.keras.Input((32, 32, 3))

    x = layers.Conv2D(C, 3, padding="same")(inputs)

    for _ in range(N):
        x = build_cell(x, arch, C)

    x = layers.GlobalAveragePooling2D()(x)

    outputs = layers.Dense(10, activation="softmax")(x)

    return tf.keras.Model(inputs, outputs)


# -----------------------------
# MAIN
# -----------------------------
api = create("/home/ankilab/hw-nats-bench/benchmarks/NATS-tss-v1_0-3ffb9-simple", 'tss', fast_mode=True, verbose=True)


arch_str = api.arch(42)

print("Architecture:")
print(arch_str)

model = build_model(arch_str)

model.summary()


# -----------------------------
# Export to TFLite
# -----------------------------

converter = tf.lite.TFLiteConverter.from_keras_model(model)

tflite_model = converter.convert()

with open("nats_model.tflite", "wb") as f:
    f.write(tflite_model)

print("TFLite model saved.")

2026-04-08 14:29:06.653762: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2026-04-08 14:29:06.657013: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2026-04-08 14:29:06.664570: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1775651346.676341 2745843 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1775651346.679861 2745843 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1775651346.690592 2745843 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linkin

[2026-04-08 12:29:08] Try to create the NATS-Bench (topology) api from /home/ankilab/hw-nats-bench/benchmarks/NATS-tss-v1_0-3ffb9-simple with fast_mode=True
[2026-04-08 12:29:08] Create NATS-Bench (topology) done with 0/15625 architectures avaliable.
Call the arch function with index=42
Architecture:
|none~0|+|nor_conv_3x3~0|nor_conv_1x1~1|+|skip_connect~0|none~1|nor_conv_3x3~2|


E0000 00:00:1775651348.285983 2745843 cuda_executor.cc:1228] INTERNAL: CUDA Runtime error: Failed call to cudaGetRuntimeVersion: Error loading CUDA libraries. GPU will not be used.: Error loading CUDA libraries. GPU will not be used.
W0000 00:00:1775651348.287749 2745843 gpu_device.cc:2341] Cannot dlopen some GPU libraries. Please make sure the missing libraries mentioned above are installed properly if you would like to use GPU. Follow the guide at https://www.tensorflow.org/install/gpu for how to download and setup the required libraries for your platform.
Skipping registering GPU devices...


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 32, 32, 3) │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d (Conv2D)     │ (None, 32, 32,    │        448 │ input_layer[0][0] │
│                     │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ multiply (Multiply) │ (None, 32, 32,    │          0 │ conv2d[0][0]      │
│                     │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add (Add)           │ (None, 32, 32,    │          0 │ multiply[0][0]    │
│                     │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_1 (Conv2D)   │ (None, 32, 32,    │      2,304 │ conv2d[0][0]      │
│                     │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_2 (Conv2D)   │ (None, 32, 32,    │        256 │ add[0][0]         │
│                     │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalization │ (None, 32, 32,    │         64 │ conv2d_1[0][0]    │
│ (BatchNormalizatio… │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 32, 32,    │         64 │ conv2d_2[0][0]    │
│ (BatchNormalizatio… │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ re_lu (ReLU)        │ (None, 32, 32,    │          0 │ batch_normalizat… │
│                     │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ re_lu_1 (ReLU)      │ (None, 32, 32,    │          0 │ batch_normalizat… │
│                     │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_1 (Add)         │ (None, 32, 32,    │          0 │ re_lu[0][0],      │
│                     │ 16)               │            │ re_lu_1[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_3 (Conv2D)   │ (None, 32, 32,    │      2,304 │ add_1[0][0]       │
│                     │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 32, 32,    │         64 │ conv2d_3[0][0]    │
│ (BatchNormalizatio… │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ multiply_1          │ (None, 32, 32,    │          0 │ add[0][0]         │
│ (Multiply)          │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ re_lu_2 (ReLU)      │ (None, 32, 32,    │          0 │ batch_normalizat… │
│                     │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_2 (Add)         │ (None, 32, 32,    │          0 │ conv2d[0][0],     │
│                     │ 16)               │            │ multiply_1[0][0], │
│                     │                   │            │ re_lu_2[0][0]     │
├─────────────────────┼───────────────────┼────────────┼─────────────────

...

 Non-trainable params: 480 (1.88 KB)

INFO:tensorflow:Assets written to: /tmp/tmp_r_e64pt/assets


INFO:tensorflow:Assets written to: /tmp/tmp_r_e64pt/assets


Saved artifact at '/tmp/tmp_r_e64pt'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 32, 32, 3), dtype=tf.float32, name='keras_tensor')
Output Type:
  TensorSpec(shape=(None, 10), dtype=tf.float32, name=None)
Captures:
  129273424444960: TensorSpec(shape=(), dtype=tf.resource, name=None)
  129273423188512: TensorSpec(shape=(), dtype=tf.resource, name=None)
  129273423337024: TensorSpec(shape=(), dtype=tf.resource, name=None)
  129273423196256: TensorSpec(shape=(), dtype=tf.resource, name=None)
  129273423339840: TensorSpec(shape=(), dtype=tf.resource, name=None)
  129273423342128: TensorSpec(shape=(), dtype=tf.resource, name=None)
  129273423338432: TensorSpec(shape=(), dtype=tf.resource, name=None)
  129273423340896: TensorSpec(shape=(), dtype=tf.resource, name=None)
  129273423334032: TensorSpec(shape=(), dtype=tf.resource, name=None)
  129273423333152: TensorSpec(shape=(), dtype=tf.resource, name=None)
  129273423331920

W0000 00:00:1775651349.847646 2745843 tf_tfl_flatbuffer_helpers.cc:365] Ignored output_format.
W0000 00:00:1775651349.847671 2745843 tf_tfl_flatbuffer_helpers.cc:368] Ignored drop_control_dependency.
2026-04-08 14:29:09.848128: I tensorflow/cc/saved_model/reader.cc:83] Reading SavedModel from: /tmp/tmp_r_e64pt
2026-04-08 14:29:09.850891: I tensorflow/cc/saved_model/reader.cc:52] Reading meta graph with tags { serve }
2026-04-08 14:29:09.850900: I tensorflow/cc/saved_model/reader.cc:147] Reading SavedModel debug info (if present) from: /tmp/tmp_r_e64pt
I0000 00:00:1775651349.878282 2745843 mlir_graph_optimization_pass.cc:425] MLIR V1 optimization pass is not enabled
2026-04-08 14:29:09.882457: I tensorflow/cc/saved_model/loader.cc:236] Restoring SavedModel bundle.
2026-04-08 14:29:10.024113: I tensorflow/cc/saved_model/loader.cc:220] Running initialization op on SavedModel bundle at path: /tmp/tmp_r_e64pt
2026-04-08 14:29:10.061136: I tensorflow/cc/saved_model/loader.cc:471] SavedModel 